# 09.04 · Skills and routers

**Module 09 · Multi-Agent Systems** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 90–120 min |
| 📋 Prerequisites | Lessons 09.01–09.03; Module 07 (`Send`, reducers). |
| 🔑 Keys | none (optional: `GROQ_API_KEY` for the 💸 cell) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Load** specialized instructions and knowledge on demand with a `load_skill` tool (skills = progressive disclosure), and measure the prompt tokens it saves.
2. **Route**: classify a request, fan out to one or more specialists **in parallel** with `Send`, and join the results.
3. **Explain** stateless routing (each turn is classified fresh) and where conversation context must come from.
4. **Swap** a keyword classifier for an LLM classifier with structured output.

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Skills](https://docs.langchain.com/oss/python/langchain/multi-agent/skills) | `load_skill`, progressive disclosure, dynamic tool registration, hierarchical skills |
| [Router](https://docs.langchain.com/oss/python/langchain/multi-agent/router) | Classify → `Command(goto=...)` or `Send` fan-out → synthesize; stateless vs stateful routers |
| [Build a multi-source knowledge base with routing](https://docs.langchain.com/oss/python/langchain/multi-agent/router-knowledge-base) | A full router with parallel sources and a synthesis step |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "langgraph>=1.2.14,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain", "langgraph",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/09-multi-agent-systems/).
import importlib.util
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Step 4: Silence known, harmless noise so real problems stand out.
import os
import warnings

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (Project 09 LIVE FAQ index): no usage telemetry.
# Same as Lesson 02.04: langchain-core 1.6.9/1.6.10 Pydantic serializer warning about the injected runtime.
warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# fastembed (Project 09 LIVE FAQ index) imports tqdm.auto; without ipywidgets it warns once. Progress bars still work.
warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

In [ ]:
# 🧪 Offline: the shared desk (Lesson 09.01 Part 2).
from langgraph.checkpoint.memory import InMemorySaver
from mastery.agents import UsageMeter
from mastery.agents.support import (SupportDesk, evaluate, faq_search, load_conversations, make_support_tools,
                                    offline_model)

desk = SupportDesk.load()
tools = make_support_tools(desk, faq_search())
conversations = load_conversations()
by_id = {c["id"]: c for c in conversations}
print(len(conversations), "conversations")

**You should now see** `15 conversations`.

---
## Part 1 · Skills: one agent, knowledge on demand

A **skill** is a named bundle of instructions and reference text. The system prompt lists only each skill's **name and one-line description**; a `load_skill(skill_name)` tool returns the full text when the agent decides it needs it. One agent, no handoffs, no subagents: the context grows only with what's used.

Our skills = each specialist's scope + that domain's FAQ file.

In [ ]:
# 🧪 Offline: the skills and how much text each one keeps OUT of the prompt until needed.
from mastery.agents.support import SKILLS_PROMPT, make_skills

skills = make_skills()
for name, skill in skills.items():
    print(f"{name:<10} {len(skill['content']):>5} chars | {skill['description']}")
listing = "\n".join(f"- {n}: {s['description']}" for n, s in skills.items())
print("\nsystem prompt:", len(SKILLS_PROMPT.format(listing=listing)), "chars")

**You should now see** three skills of about 1,150–1,500 characters each, and a system prompt of about 590 characters that only **lists** them.

In [ ]:
# 🧪 Offline: a repeat request (S12) with skills: watch load_skill happen once.
from mastery.agents.support import build_skills_desk, run_conversation

skills_desk = build_skills_desk(offline_model(), tools, checkpointer=InMemorySaver())
meter = UsageMeter()
outcome = run_conversation(skills_desk, by_id["S12"], desk, callbacks=[meter], thread_id="skills-1")
state = skills_desk.get_state({"configurable": {"thread_id": "skills-1"}}).values
print([c["name"] for m in state["messages"] if m.type == "ai" for c in m.tool_calls])
print("answer:", outcome["answer"])
print(meter.summary())

**You should now see** the tool calls `['load_skill', 'lookup_invoices', 'issue_refund']` (the skill loaded once, in turn 1, and still in the history for turn 2), the refund answer, and **5** model calls, the docs' number for skills on a repeat request.

In [ ]:
# 🧪 Offline: skills vs pasting every skill into the prompt up front ("inline").
rows = []
for label, inline in (("skills (load_skill)", False), ("inline (all in prompt)", True)):
    row = {"design": label}
    for conv_id in ("S01", "S12", "S14"):
        meter = UsageMeter()
        evaluate(build_skills_desk(offline_model(), tools, checkpointer=InMemorySaver(), inline=inline),
                 [by_id[conv_id]], desk, meter=meter)
        row[conv_id] = f"{meter.model_calls} calls / {meter.input_tokens:,} in"
    rows.append(row)
table(rows, ["design", "S01", "S12", "S14"])

**You should now see** skills at 3 / 5 / 5 calls and inline at 2 / 4 / 3 calls. Inline uses **more input tokens** on S01 and S12 (about 3.4K vs 3.0K, 6.9K vs 5.4K: every call carries all three FAQs), but **fewer** on S14 (about 5.2K vs 6.2K), where skills paid for two loads and two extra calls.

### What just happened?

Skills trade **one extra call per skill** (the load) for a smaller prompt on every call. With three short FAQs the saving is modest, and a request that needs several skills can even cost more; with 30 skills of 5,000 tokens each, pasting everything is not even possible. The docs' extensions build on this: register a skill's **tools** only when it's loaded (dynamic tools), or nest skills (a skill that lists sub-skills). The cost: one agent still does everything, so instructions from different skills can still clash in one context.

---
## Part 2 · The router: classify, fan out, join

A router is a small graph: `route` classifies the latest message, `Send` starts one branch per matching specialist (**in parallel**), and `join` combines the answers. Here the classifier is keyword code (`detect_intents`) and the join is code too, so the only model calls are the specialists'.

In [ ]:
# 🧪 Offline: the router graph and a two-domain request (S14).
from mastery.agents.support import build_router_desk, detect_intents, split_by_intent

router = build_router_desk(offline_model, tools, checkpointer=InMemorySaver())
print("nodes:", [n for n in router.get_graph().nodes if not n.startswith("__")])
text = by_id["S14"]["turns"][0]
print("intents:", detect_intents(text))
print("clauses:", split_by_intent(text))

desk.reset()
meter = UsageMeter()
out = router.invoke({"messages": [{"role": "user", "content": text}]},
                    {"configurable": {"thread_id": "router-1"}, "callbacks": [meter]})
print("routes:", out["routes"], "| calls:", {k: v["model_calls"] for k, v in meter.agents().items()})
print("answer:", out["messages"][-1].content[:160])

**You should now see** `nodes: ['route', 'billing', 'technical', 'account', 'join']`, `intents: ['billing', 'technical']`, one clause per domain, `routes: ['billing', 'technical']` with 2 calls each, and an answer that starts with the refund and continues with the `API errors and rate limits` passage.

### What just happened?

`route` returned `[Send("billing", {...}), Send("technical", {...})]`; both branches ran in the **same superstep** (in parallel) and wrote into `results`, a list with a reducer (`_collect`, which appends, and clears when a new turn starts). `join` ran once, after both. No coordinator call before or after: that's why the router is the cheapest pattern on this desk (Lesson 09.01: 36 calls, ~13K tokens).

---
## Part 3 · Stateless routing: where does context come from?

Each turn is classified **fresh**: the router doesn't remember that the last turn was billing, and specialists get only the routed query. Two consequences, both visible on the test set:

- **S15** (`"Hello, I need some help."` → `"It's about a refund: ..."`): turn 1 matches nothing → `join` asks a clarifying question with **zero** model calls.
- **S12** turn 2 has no customer ID: the router adds `(customer C-1001)` from earlier turns. That's **code** carrying context, the router's version of 09.02's coordinator prompt rule.

In [ ]:
# 🧪 Offline: S15 and S12, turn by turn.
for conv_id in ("S15", "S12"):
    desk.reset()
    config = {"configurable": {"thread_id": f"router-{conv_id}"}}
    for turn in by_id[conv_id]["turns"]:
        meter = UsageMeter()
        out = router.invoke({"messages": [{"role": "user", "content": turn}]}, {**config, "callbacks": [meter]})
        print(f"{conv_id} routes={out['routes']!s:<12} calls={meter.model_calls} | {out['messages'][-1].content[:75]}")

**You should now see** S15: `routes=[]`, `calls=0`, the clarifying question; then `routes=['billing']`, `calls=2`, `Refunded $19.00 ...`. S12: `['billing']` twice with 2 calls each (invoices, then `Refunded $49.00 for INV-1003 ...`).

> The docs' **stateful** variant wraps the whole router as a **tool** of a conversational agent: the agent keeps the conversation and decides when to call the router. That's Lesson 09.02's pattern with a router inside, and it costs the agent's calls again.

---
## Part 4 · A model as the classifier

> 💸 Calls `gpt-oss-20b` several times (multi-agent = many small calls), a few thousand tokens of your free-tier quota. Without `GROQ_API_KEY` it prints `⏭️ Skipped`.

In [ ]:
# 💸 LIVE: an LLM classifier (structured output) plugged into the same router.
from typing import Literal

from pydantic import BaseModel, Field


class Routes(BaseModel):
    """Which specialists must handle the customer's latest message (empty if unclear)."""

    domains: list[Literal["billing", "technical", "account"]] = Field(default_factory=list)


if HAVE_GROQ_KEY:
    from mastery.models import get_model

    classifier = get_model("fast", temperature=0, reasoning_effort="low").with_structured_output(Routes)

    def llm_classify(text: str) -> list[str]:
        return classifier.invoke([
            ("system", "Classify an Orbita Cloud support message. billing = invoices, charges, refunds; "
                       "technical = API errors, sync, webhooks, status; account = passwords, locks, emails."),
            ("user", text)]).domains

    for conv_id in ("S06", "S11", "S14", "S15"):
        text = by_id[conv_id]["turns"][0]
        print(f"{conv_id} keywords={detect_intents(text)!s:<26} llm={llm_classify(text)}")
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** (with a key) both classifiers agreeing on S06 (`technical`), S11 (`account`) and S14 (`billing` + `technical`); for S15 (`Hello, I need some help.`) the keywords give `[]` and the model usually also returns `[]`. Plug it in with `build_router_desk(..., classify=llm_classify)`: one extra model call per turn, in exchange for understanding requests the keywords miss ("I was double-billed", "the export link expired"). Without a key: `⏭️ Skipped`.

---
## Part 5 · Skills and router on the test set

In [ ]:
# 🧪 Offline: 15 conversations, skills vs router.
rows = []
for name, build in {"skills": lambda: build_skills_desk(offline_model(), tools, checkpointer=InMemorySaver()),
                    "router": lambda: build_router_desk(offline_model, tools, checkpointer=InMemorySaver())}.items():
    meter = UsageMeter()
    report = evaluate(build(), conversations, desk, meter=meter)
    rows.append({"pattern": name, "passed": f"{report['passed']}/15", "calls": meter.model_calls,
                 "input tokens": f"{meter.input_tokens:,}"})
table(rows, ["pattern", "passed", "calls", "input tokens"])

**You should now see** `skills  15/15  53` calls with about 55K input tokens, and `router  15/15  36` calls with about 12K. Skills carry the loaded FAQ text in every later call of the conversation; the router's specialists start fresh each turn.

---
## ⚠️ Pitfalls and troubleshooting (Windows Git Bash included)

| Symptom | Cause | Fix |
|---|---|---|
| The agent answers without loading the skill | The prompt doesn't make loading mandatory, or the description is vague | "Before you handle a request, load the matching skill"; precise one-line descriptions |
| Skill loaded on every turn | The agent doesn't see earlier tool results (history trimmed) | Keep `load_skill` results in context, or remember loaded skills in state |
| Router results from the last turn show up again | A plain `operator.add` reducer on `results` | Clear per turn (the router's `_collect` reducer resets on `None`) |
| `InvalidUpdateError` when two branches write the same key | No reducer on a key that parallel `Send` branches write | Add a reducer (`Annotated[list, ...]`) |
| Router misroutes "I was double-billed" | Keyword classifier | LLM classifier (Part 4), or add the phrase to the rules and to the test set |
| Specialist asks for the customer ID | Stateless routing; query without context | Add context in `fan_out` (the router adds the customer ID) |

## ⚠️ Production notes

> ⚠️ **Production note: a router is the cheapest multi-agent pattern when classification is easy.**
> Code classifiers are free and deterministic; test them like code (the 15 conversations are a start). Add an LLM classifier only for the misses, and log every route for review.

> ⚠️ **Production note: skills are content; version them.**
> Skill texts are prompts plus knowledge. Keep them in files under version control, review changes like code, and re-run the eval when one changes.

## 🏋️ Exercises

1. **(Easy)** Add an `"onboarding"` skill (a short text about setting up a first station) to `make_skills()`'s result and load it by hand: `load_skill_tool(skills).invoke({"skill_name": "onboarding"})`.
2. **(Medium)** Make the router stateful the docs' way: wrap `router` as a tool (`@tool def support_router(query: str) -> str`) of a conversational `create_agent`. Count calls on S12.
3. **(Medium)** Replace the code `join` with a model call that writes one friendly paragraph. What does it cost on the test set (offline: count the calls you add)?
4. **(Stretch)** Dynamic tools: register each domain's tools only after its skill is loaded (middleware that filters `request.tools` by loaded skills, like 09.03's `step_middleware`).

<details>
<summary><b>Show solutions</b></summary>

**1.**
```python
from mastery.agents.support import load_skill_tool
skills["onboarding"] = {"description": "Setting up a first station.",
                        "content": "# Skill: onboarding\nPair the station in the app, then ..."}
print(load_skill_tool(skills).invoke({"skill_name": "onboarding"})[:60])
```

**2.**
```python
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def support_router(query: str) -> str:
    '''Route a customer request to the right specialists and return their answers.'''
    out = router.invoke({"messages": [{"role": "user", "content": query}]}, {"configurable": {"thread_id": "inner"}})
    return out["messages"][-1].content
```
Give the agent a checkpointer; the router then needs no checkpointer of its own (pass none, or a fresh thread per call). Each turn costs the agent's 2 calls + the router's.

**3.** One extra call per turn: 36 + 16 turns ≈ 52 calls offline, close to handoffs. The docs' router does this (their 3-call one-shot = classify + agent + synthesize).

**4.** In a `@wrap_model_call`, read the `load_skill` ToolMessages from `request.messages`, compute the allowed tool names, and `handler(request.override(tools=[t for t in request.tools if t.name in allowed]))`. Register all tools on `create_agent`, as in 09.03.
</details>

## ✅ Checkpoint

- [ ] I built a skills agent with `load_skill` and measured its prompt tokens against an inline prompt.
- [ ] I built a router with parallel `Send` branches and a reducer that resets per turn.
- [ ] I can explain stateless routing and how context reaches the specialists.
- [ ] I know how to swap in an LLM classifier and what it costs.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Read [Skills](https://docs.langchain.com/oss/python/langchain/multi-agent/skills) and note in [`docs/research-log.md`](../../docs/research-log.md) which extensions the docs list today (on 2026-10-12: dynamic tool registration, hierarchical skills, reference awareness) and whether the skills format is shared with Deep Agents (Module 11).

## Summary

- Skills: one agent, a short prompt that lists skills, `load_skill` loads instructions + knowledge on demand (3 / 5 calls one-shot / repeat, like the docs).
- Router: classify → `Send` fan-out (parallel) → join; cheapest here (36 calls) because classification and joining are code.
- Routers are stateless per turn: code (or a wrapping agent) must carry context.
- An LLM classifier costs one call per turn and catches what keywords miss.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 09.05 · Custom workflows and choosing an architecture** → [`09-05-custom-workflow-and-architecture.ipynb`](09-05-custom-workflow-and-architecture.ipynb).

[Module 09 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)